In [1]:
"""
County data used for clustering.
"""
import pandas as pd
fips_to_state = {
    '01': 'AL', '02': 'AK', '04': 'AZ', '05': 'AR', '06': 'CA',
    '08': 'CO', '09': 'CT', '10': 'DE', '11': 'DC', '12': 'FL',
    '13': 'GA', '15': 'HI', '16': 'ID', '17': 'IL', '18': 'IN',
    '19': 'IA', '20': 'KS', '21': 'KY', '22': 'LA', '23': 'ME',
    '24': 'MD', '25': 'MA', '26': 'MI', '27': 'MN', '28': 'MS',
    '29': 'MO', '30': 'MT', '31': 'NE', '32': 'NV', '33': 'NH',
    '34': 'NJ', '35': 'NM', '36': 'NY', '37': 'NC', '38': 'ND',
    '39': 'OH', '40': 'OK', '41': 'OR', '42': 'PA', '44': 'RI',
    '45': 'SC', '46': 'SD', '47': 'TN', '48': 'TX', '49': 'UT',
    '50': 'VT', '51': 'VA', '53': 'WA', '54': 'WV', '55': 'WI',
    '56': 'WY'
}


In [2]:
education = pd.read_excel('Education_wo_xcol.xlsx', skiprows=3)
population = pd.read_excel('PopulationEstimates_wo_extra_col.xlsx', skiprows=4)
poverty = pd.read_excel('PovertyEstimates1.xlsx', skiprows=4)
unemployment = pd.read_excel('Unemployment_wo_xcol.xlsx', skiprows=4)
location = pd.read_csv("us_county_latlng.csv")
race = pd.read_csv("county_race_data.csv")

In [3]:
# import requests
# import pandas as pd

# # Census API Key (Replace with your own key if you have one)
# CENSUS_API_KEY = "d9881a8581208722b20214826b5546416262b82a"

# # Define the base API URL
# BASE_URL = "https://api.census.gov/data/2022/acs/acs5"

# # Variables to fetch (Total population + breakdown by race)
# variables = {
#     "total_pop": "B02001_001E",  # Total Population
#     "white": "B02001_002E",      # White alone
#     "black": "B02001_003E",      # Black or African American alone
#     "native": "B02001_004E",     # American Indian and Alaska Native alone
#     "asian": "B02001_005E",      # Asian alone
#     "pacific": "B02001_006E",    # Native Hawaiian and Other Pacific Islander alone
#     "other": "B02001_007E",      # Some Other Race alone
#     "two_or_more": "B02001_008E" # Two or More Races
# }

# # Construct the API request URL
# params = {
#     "get": ",".join(variables.values()),
#     "for": "county:*",
#     "in": "state:*",
#     "key": CENSUS_API_KEY
# }

# response = requests.get(BASE_URL, params=params)

# if response.status_code != 200:
#     print("Error: API request failed with status code", response.status_code)
#     print("Response text:", response.text)
#     exit()

# try:
#     data = response.json()
# except requests.exceptions.JSONDecodeError:
#     print("Error: Unable to decode JSON response")
#     print("Response text:", response.text)
#     exit()


# # Fetch the data
# # response = requests.get(BASE_URL, params=params)
# # data = response.json()

# # Convert data to DataFrame
# columns = ["total_pop", "white", "black", "native", "asian", "pacific", "other", "two_or_more", "state", "county"]
# df = pd.DataFrame(data[1:], columns=columns)

# # Convert numeric columns to integers
# for col in columns[:-2]:
#     df[col] = df[col].astype(int)

# # Add FIPS code column (state + county)
# df["county_fips"] = df["state"] + df["county"]

# # Convert counts to percentages
# race_cols = ["white", "black", "native", "asian", "pacific", "other", "two_or_more"]
# # for col in race_cols:
# #     df[col + "_pct"] = (df[col] / df["total_pop"]) * 100

# # Keep relevant columns
# # df = df[["county_fips", "total_pop"] + [col + "_pct" for col in race_cols]]

# # Save to CSV
# df.to_csv("county_race_data.csv", index=False)

# print("Race data for all counties saved to county_race_data.csv!")


In [4]:
education.rename(columns={'FIPS Code': 'FIPS', 'Area name': 'Area_Name'}, inplace=True)
population.rename(columns={'FIPStxt': 'FIPS'}, inplace=True)
poverty.rename(columns={'FIPS_Code': 'FIPS', 'Area_name': 'Area_Name'}, inplace=True)
unemployment.rename(columns={'FIPS_Code': 'FIPS'}, inplace=True)
location.rename(columns={'fips_code': 'FIPS', 'name': 'Area_Name'}, inplace=True)
race.rename(columns={'county_fips': 'FIPS'}, inplace=True)

population = population.drop(columns=['Area_Name'])
poverty = poverty.drop(columns=['Area_Name'])
unemployment = unemployment.drop(columns=['Area_Name'])
location = location.drop(columns=['Area_Name'])
race = race.drop(columns=['county', 'state'])

In [5]:
county_characteristics = [education, population, poverty, unemployment, location, race]
for i, df in enumerate(county_characteristics):
    print(f"DataFrame {i + 1} FIPS dtype: {df['FIPS'].dtype}")
    df['FIPS'] = df['FIPS'].astype(str).str.zfill(5)
    print(df.columns)

location['State'] = location['FIPS'].str[:2].map(fips_to_state)
race['State'] = race['FIPS'].str[:2].map(fips_to_state)

# the combined dataframe
county_dataframe = county_characteristics[0]
for df in county_characteristics[1:]:
    county_dataframe = county_dataframe.merge(df, on=['FIPS', 'State'], how='outer')


columns_to_drop = ['INTERNATIONAL_MIG_2023',
                   'Less than a high school diploma, 2018-22', 'High school diploma only, 2018-22', "Some college or associate's degree, 2018-22",
                   "Bachelor's degree or higher, 2018-22", 'DOMESTIC_MIG_2023', 'R_BIRTH_2023', 'R_DEATH_2023', 'R_INTERNATIONAL_MIG_2023', 'R_DOMESTIC_MIG_2023',
                   'R_NET_MIG_2023', 'CI90LBALL_2021', 'CI90UBALL_2021','CI90LBALLP_2021', 'CI90UBALLP_2021','CI90LB017_2021', 'CI90UB017_2021', 'CI90LB017P_2021',
                   'CI90UB017P_2021', 'CI90LB517_2021', 'CI90UB517_2021', 'CI90LB517P_2021', 'CI90UB517P_2021','CI90LBINC_2021', 'CI90UBINC_2021', 'CI90LB04_2021',
                   'CI90UB04_2021', 'CI90LB04P_2021', 'CI90UB04P_2021', 'Metro_2013', 'Employed_2022','Unemployed_2022', 'BIRTHS_2023','RESIDUAL_2023',
                   'POV017_2021', 'PCTPOV017_2021', 'POV517_2021', 'PCTPOV517_2021', 'POV04_2021', 'PCTPOV04_2021',
                   'Rural_Urban_Continuum_Code_2013']
county_dataframe = county_dataframe.drop(columns=columns_to_drop)
df_ca = county_dataframe[county_dataframe['FIPS'].astype(str).str.zfill(5).str.startswith('06')]

county_dataframe.to_pickle("county_data.pkl")
df_ca.to_pickle("county_data_ca.pkl")

print(county_dataframe.columns)

DataFrame 1 FIPS dtype: int64
Index(['FIPS', 'State', 'Area_Name', '2023 Rural-urban Continuum Code',
       'Less than a high school diploma, 2018-22',
       'High school diploma only, 2018-22',
       'Some college or associate's degree, 2018-22',
       'Bachelor's degree or higher, 2018-22',
       'Percent of adults with less than a high school diploma, 2018-22',
       'Percent of adults with a high school diploma only, 2018-22',
       'Percent of adults completing some college or associate's degree, 2018-22',
       'Percent of adults with a bachelor's degree or higher, 2018-22'],
      dtype='object')
DataFrame 2 FIPS dtype: int64
Index(['FIPS', 'State', 'POP_ESTIMATE_2023', 'BIRTHS_2023', 'NATURAL_CHG_2023',
       'INTERNATIONAL_MIG_2023', 'DOMESTIC_MIG_2023', 'NET_MIG_2023',
       'RESIDUAL_2023', 'GQ_ESTIMATES_2023', 'R_BIRTH_2023', 'R_DEATH_2023',
       'R_NATURAL_CHG_2023', 'R_INTERNATIONAL_MIG_2023', 'R_DOMESTIC_MIG_2023',
       'R_NET_MIG_2023'],
      dtype='objec